# A4 — PINN Digital Twin — Methanol Reactor

## Approach (how this PINN is built)

A **Physics-Informed Neural Network (PINN)** does **not** integrate an ODE with solve_ivp.  
It learns a continuous profile ŷ(z, u) with a neural network, and the ODE is enforced as a **loss term**.

`	ext
Inputs:  u = (T_in, P_in, F_in),  z ∈ [0, L]
Network: NN(z, u) → states along the bed
Loss:    L = L_data + λ_phys · L_physics + λ_bc · L_BC
Twin:    query NN at z = 0.2…1.0 m → 8 Excel-style outputs
`

### Build steps in this notebook

1. **Load DATA_clean.csv** — flip the heat-input sign, convert conversion from a fraction to percent, and drop cases with inlet temperature above 275 °C.
2. **Define PINN** — NN(z, u) with **hard inlet BC** at z=0 and stoichiometry for CO/H₂.
3. **Learnable kinetics scale** — η(u) so LHHW does not over-convert like the raw physics ODE.
4. **Physics residual** — autodiff dŷ/dz vs mass / energy / Ergun / heat RHS (Torch).
5. **Train** — minimize data match (Excel T, P, Q, X) + light physics residual.
6. **Digital twin** — predict_outputs(T_in, P_in, F_in) → 8 outputs + interactive cell.

| Loss | Meaning |
|------|--------|
| **L_data** | Match Excel T at 0.2–1.0 m, P_out, Q, conversion |
| **L_physics** | Autodiff dŷ/dz should equal ODE right-hand side |
| **L_BC** | Soft check that z=0 matches the inlet (also hard-coded) |


**Reaction:** CH₃OH → CO + 2H₂  
**Data:** DATA_clean.csv  

Note: Excel T/Q disagree with strong wall heating at low conversion, so training is **data-dominant** (λ_phys small). Raise N_TRAIN_CASES / EPOCHS for better accuracy.
#
This note means the model's training will prioritize fitting the data (Excel T/Q) over enforcing the physics loss, since the data and physics sometimes disagree (especially at low conversion). As a result, λ_phys is set small, making the PINN behave more like a standard neural net regressor than a physics-driven model in those regions. Raising N_TRAIN_CASES/EPOCHS helps the network fit data better.


## 1. Imports

In [ ]:
from pathlib import Path

# Import necessary libraries
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split

# Set up file paths 
DATA_FILE = Path("DATA_clean.csv")
T_heat_C = 275.0  # heating medium [°C]; inlet-temperature filter for DATA_clean.csv


# Set random seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)

# Positions along the reactor bed (meters)
MEASUREMENT_POSITIONS_M = np.array([0.2, 0.4, 0.6, 0.8, 1.0])
P_ATM_BAR = 1.01325  # Atmospheric pressure (bar)

# Input and output columns expected from Excel
INPUT_COLS = ["T_in_C", "P_in_bar_g", "F_in_kgmole_h"]
OUTPUT_COLS = [
    "T_0.2m_C", "T_0.4m_C", "T_0.6m_C", "T_0.8m_C", "T_1m_C",
    "P_out_bar_g", "Q_total_kW", "X_overall_pct",
]
TEMP_COLS = ["T_0.2m_C", "T_0.4m_C", "T_0.6m_C", "T_0.8m_C", "T_1m_C"]

# Print some environment info
print("Torch:", torch.__version__, )
print("Data file:", DATA_FILE.resolve())


## 2. Load and clean `DATA_clean.csv`


In [ ]:
def load_data(path: Path) -> pd.DataFrame:
    # DATA_clean.csv already uses the short column names.
    # Cleaning matches revised_Physics Model_1.
    data = pd.read_csv(path)
    n_raw = len(data)

    data["Q_total_kW"] = -data["Q_total_kW"]
    # CSV stores heat into the reactor as negative; the model uses positive heat input.

    data["X_overall_pct"] = data["X_overall_pct"] * 100.0
    # CSV stores conversion as a fraction.

    too_hot = data["T_in_C"] > T_heat_C
    print(f"Total rows in file          : {n_raw}")
    print(f"Removed (T_in > {T_heat_C:.0f} °C)     : {int(too_hot.sum())}")

    ok = (
        ~too_hot
        & data["X_overall_pct"].between(0, 100)
        & data["Q_total_kW"].between(-50, 50)
        & data[["T_in_C", "P_in_bar_g", "F_in_kgmole_h"]].notna().all(axis=1)
    )
    cleaned = data.loc[ok].reset_index(drop=True)
    print(f"Valid rows                  : {len(cleaned)}")
    return cleaned

df = load_data(DATA_FILE)  # Main dataframe; full cleaned dataset
# Split dataset into training and test sets (80%/20%)
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)
train_df = train_df.reset_index(drop=True)  # Reset indices after split
test_df = test_df.reset_index(drop=True)
print(f"Train={len(train_df)}  Test={len(test_df)}")
display(df[INPUT_COLS + OUTPUT_COLS].head())  # Show a preview of the processed data


## 3. Reactor constants and scaling

PINNs need scaled inputs/outputs so residuals stay numerically stable.

Thermo (same as A3 / A2 / Parallel):
- **R** = `1.987` kcal/(kgmole·K)
- **ΔH(25 °C)** = `2.2e+04` kcal/kgmole
- **ΔH(T)** = ΔH(25°C) + ΔCp·(T − 25)  (temperature-dependent)


In [ ]:
# --- Thermo constants (same as A3 / A2 / Parallel) ---
R_kcal = 1.987                 # gas constant [kcal/(kgmole·K)]
delta_H_ref_kcal = 2.2e4       # reaction heat at 25 °C [kcal/kgmole]
T_ref_C = 25.0                 # reference temperature [°C]

# Convert to SI for physics residual: kcal/kgmole -> J/mol
KCAL_PER_KGMOLE_TO_J_PER_MOL = 4184.0 / 1000.0
R = R_kcal * KCAL_PER_KGMOLE_TO_J_PER_MOL                      # [J/(mol·K)]
delta_H_ref = delta_H_ref_kcal * KCAL_PER_KGMOLE_TO_J_PER_MOL  # [J/mol] at 25 °C

# Reactor inner diameter (meters)
D = 0.1

# Reactor length (meters)
L = 1.0

# Jacket-side coefficient h_o [W/(m^2·K)]. The residual uses h_heat = U(F_in).
h_o = 115.0

# External heat source temperature (Celsius)
T_heat_C = 275.0

# Reactor porosity (fraction of reactor volume that is void)
epsilon = 0.33

# Catalyst particle diameter (meters)
dp = 1.0e-3

# Reactor cross-sectional area (m^2)
Ac = np.pi * D**2 / 4

# Reactor inner perimeter (meters)
perimeter = np.pi * D

# Molar mass (kg/mol) for each component:
MW_CH3OH = 32.04e-3   # Methanol
MW_CO = 28.01e-3      # Carbon monoxide
MW_H2 = 2.016e-3      # Hydrogen

# Molar heat capacities (J/(mol·K)) for each component
Cp_CH3OH = 43.0
Cp_CO = 29.0
Cp_H2 = 29.0

# ΔCp for reaction CH3OH -> CO + 2H2  [J/(mol·K)]
delta_Cp = Cp_CO + 2.0 * Cp_H2 - Cp_CH3OH

# Arrhenius pre-factor for overall methanol reaction
A_kinetic = 7.0e10

# Activation energy [kcal/kgmole]; Arrhenius uses R_kcal
E_kinetic = 19120.0

# Arrhenius pre-factor for CO adsorption
A_CO = 1.0e7

# CO adsorption energy [kcal/kgmole]; Arrhenius uses R_kcal
E_CO = 6036.8



mu_gas = 2.0e-5          # Gas viscosity [Pa·s]
k_gas = 0.035            # Gas thermal conductivity [W/(m·K)]


def bed_side_htc(F_in_mol_s):
    # Specchia bed-side wall coefficient h_i [W/(m²·°C)]
    mass_flux = F_in_mol_s * MW_CH3OH / Ac
    Re_p = dp * mass_flux / mu_gas
    Nu_w = 2.0 * epsilon + 0.0835 * (Re_p ** 0.91)
    return Nu_w * k_gas / dp


def overall_U(F_in_mol_s):
    # Overall wall coefficient. h_heat = U, from the inlet methanol flow.
    h_i = bed_side_htc(F_in_mol_s)
    return 1.0 / (1.0 / h_o + 1.0 / h_i)

def reaction_enthalpy(T_C):
    # Temperature-dependent ΔH [J/mol]; works with float or torch tensor
    # ΔH(T) = ΔH(25°C) + ΔCp*(T-25)
    return delta_H_ref + delta_Cp * (T_C - T_ref_C)


# Scaling references from training data
T_REF = float(train_df["T_in_C"].mean())
P_REF_BAR = float(train_df["P_in_bar_g"].mean() + P_ATM_BAR)
F_REF = float((train_df["F_in_kgmole_h"] * 1000.0 / 3600.0).mean())
Q_REF = 1.0

print(f"R = {R_kcal} kcal/(kgmole·K)  ->  {R:.6f} J/(mol·K)")
print(f"ΔH(25°C) = {delta_H_ref_kcal:.3e} kcal/kgmole  ->  {delta_H_ref:.3e} J/mol")
print(f"ΔCp = {delta_Cp:.2f} J/(mol·K)")
print(f"T_REF={T_REF:.1f} °C, P_REF={P_REF_BAR:.3f} bar, F_REF={F_REF:.4e} mol/s")


## 4. PINN network

- Input: `(z/L, T_in_n, P_in_n, F_in_n)`
- Output head: raw network features
- **Hard inlet BC(Boundary Conditions):** states are written so `z=0` exactly matches the inlet
- Stoichiometry: `F_CO = F_in − F_CH3OH`, `F_H2 = 2(F_in − F_CH3OH)`
- Learnable rate scale: `η(u) = softplus(η_net(u))` (small MLP)

In this step, we begin defining the Physics-Informed Neural Network (PINN) architecture that will emulate the reactor’s behavior. The PINN takes normalized spatial and process input variables and predicts the reactor’s internal states along its length. Setting up this network is essential for leveraging both experimental data and physical laws to accurately model and infer the system’s behavior under varying operating conditions.

In [ ]:
class ReactorPINN(nn.Module):
    """Physics-Informed Neural Network for reactor simulation with hard-coded inlet BC."""
    # This PINN uses a hard-coded inlet boundary condition (BC) as follows:
    # At the reactor inlet (z=0), the state values are set directly from the inlet process variables:
    #   - T (temperature) is set to the inlet temperature, T_in [C]
    #   - P (pressure) is set to the inlet absolute pressure, P_in [bar abs] = P_in_bar_g + P_ATM_BAR
    #   - F (total molar flow) is set to the inlet flow, F_in [mol/s] = F_in_kgmole_h * 1000 / 3600
    #   - Q (heat flow) is set to zero at the inlet, Q=0
    # This ensures that the initial state of the reactor at z=0 matches the specified process inlet conditions.

    def __init__(self, hidden=64, depth=4):
        super().__init__()
        # Main network for state derivatives: input=[z_norm, T_in_norm, P_in_norm, F_in_norm]
        input_size = 4
        output_size = 4  # [dF, dT, dP, dQ]
        layers = [nn.Linear(input_size, hidden), nn.Tanh()]
        for _ in range(depth - 1):
            layers += [nn.Linear(hidden, hidden), nn.Tanh()]
        layers.append(nn.Linear(hidden, output_size))
        self.net = nn.Sequential(*layers)

        # Small network for learnable rate scaling eta(u)
        eta_input_size = 3  # [T_in_norm, P_in_norm, F_in_norm]
        eta_hidden = 32
        self.eta_net = nn.Sequential(
            nn.Linear(eta_input_size, eta_hidden),
            nn.Tanh(),
            nn.Linear(eta_hidden, 1),
        )
        # Initialize eta_net bias to start with a low conversion rate
        with torch.no_grad():
            self.eta_net[-1].bias.fill_(0.541324854)  # softplus ≈ 1 for the revised rate law

    def eta(self, u_n):
        # Compute positive rate scaling eta from normalized process inputs
        # u_n: (N, 3)
        return torch.nn.functional.softplus(self.eta_net(u_n)) + 1e-12

    def forward(self, z, u_phys):
        """
        Returns predicted states at each position along reactor.
        z: (N, 1) Position in meters
        u_phys: (N, 3) = [T_in_C, P_in_bar_g, F_in_kgmole_h]
        Returns dict of state tensors all shape (N, 1)
        """
        # Unpack input variables
        T_in = u_phys[:, 0:1]                  # Inlet temp [C]
        P_in_bar_g = u_phys[:, 1:2]            # Inlet pressure [bar gauge]
        F_in_kgmole_h = u_phys[:, 2:3]         # Inlet flow [kgmole/h]

        # Convert units for normalization and physics
        F_in = F_in_kgmole_h * 1000.0 / 3600.0    # mol/s
        P_in_Pa = (P_in_bar_g + P_ATM_BAR) * 1.0e5   # Pa

        # Normalize inputs for neural net
        z_n = z / L  # normalized position (0 to 1)
        T_in_n = T_in / T_REF
        P_in_n = (P_in_bar_g + P_ATM_BAR) / P_REF_BAR
        F_in_n = F_in / F_REF
        u_n = torch.cat([T_in_n, P_in_n, F_in_n], dim=1)  # (N, 3)
        # Network input: concat position and normalized process vars
        x = torch.cat([z_n, u_n], dim=1)  # (N, 4)

        # Neural net predicts raw rates of change/features
        raw = self.net(x)  # (N, 4)

        # Hard inlet boundary: ensure z=0 recovers feed exactly

        # Methanol consumption along reactor;
        # Use softplus to force non-negative consumption (no production)
        methanol_consumed = z_n * torch.nn.functional.softplus(raw[:, 0:1]) * F_in  # (N,1)
        F_CH3OH = F_in - methanol_consumed
        F_CH3OH = torch.clamp(F_CH3OH, min=1e-12)  # Avoid negative

        # CO produced equals methanol consumed; H2 is twice this (stoichiometry)
        F_CO = methanol_consumed
        F_H2 = 2.0 * methanol_consumed

        # Temperature rise along the reactor (arbitrary 100x for scaling)
        T_C = T_in + z_n * raw[:, 1:2] * 100.0

        # Pressure drop along the reactor, forced to be positive
        P_drop = z_n * torch.nn.functional.softplus(raw[:, 2:3]) * (0.2 * 1e5)
        P = P_in_Pa - P_drop
        P = torch.clamp(P, min=1.0e3)  # min 1 kPa for stability

        # Heat transferred/byproduct heat (arbitrary scaling)
        Q_kW = z_n * raw[:, 3:4] * Q_REF

        # Return all relevant states
        return {
            "F_CH3OH": F_CH3OH,  # Methanol flow (mol/s)
            "F_CO": F_CO,        # CO flow (mol/s)
            "F_H2": F_H2,        # H2 flow (mol/s)
            "T_C": T_C,          # Temperature (C)
            "P": P,              # Pressure (Pa)
            "Q_kW": Q_kW,        # Heat transfer (kW)
            "F_in": F_in,        # Inlet flow (mol/s)
            "u_n": u_n,          # Normalized process vars
        }


# Instantiate and print summary
model = ReactorPINN()
print(model)
print("Parameters:", sum(p.numel() for p in model.parameters()))


## 5. Physics residual (ODE RHS in Torch)

Autodiff gives `dŷ/dz`. Physics loss penalizes mismatch with mass / energy / pressure / heat ODEs.
#
In this step, we define the physics-based residual that represents the right-hand side (RHS) of the ordinary differential equations (ODEs) using PyTorch operations. Autodifferentiation will be used to compute the derivatives of the predicted states with respect to the reactor position, and the physics loss will penalize any mismatch between these derivatives and the corresponding ODEs for mass, energy, pressure, and heat transfer.

In [ ]:
def reaction_rate_torch(T_C, P, F_CH3OH, F_CO, F_H2, eta):
    # Convert temperature to Kelvin
    T_abs = T_C + 273.15

    # Total flow to avoid zero division
    F_total = torch.clamp(F_CH3OH + F_CO + F_H2, min=1e-12)

    # Compute total concentration (ideal gas law)
    C_total = P / (R * T_abs) / 1000.0  # kgmole/m³

    # Molar fractions times total conc. give partial concentrations
    C_M = (F_CH3OH / F_total) * C_total
    C_CO = (F_CO / F_total) * C_total

    # Reaction numerator (Arrhenius law). E and R_kcal are in kcal/kgmole.
    num = A_kinetic * torch.exp(-E_kinetic / (R_kcal * T_abs)) * C_M

    # Reaction denominator (CO inhibition)
    den = 1.0 + A_CO * torch.exp(-E_CO / (R_kcal * T_abs)) * C_CO

    # mol/(m³·s), then scaled by the learned effectiveness eta
    return eta * (num / den) * 1000.0


def ode_rhs(states, eta):
    # Unpack state variables from dictionary
    F_CH3OH = states["F_CH3OH"]  # methanol flow
    F_CO = states["F_CO"]        # CO flow
    F_H2 = states["F_H2"]        # H2 flow
    T_C = states["T_C"]          # Temperature (C)
    P = states["P"]              # Pressure (Pa)

    # Compute reaction rate (arrhenius law, inhibition, effectiveness)
    r = reaction_rate_torch(T_C, P, F_CH3OH, F_CO, F_H2, eta)
    rpl = r * Ac                 # Reaction per length (mol/s/m)

    # Compute heat transferred from wall to gas
    h_heat = overall_U(states["F_in"])  # revised physics: U from inlet methanol flow
    q_transfer = h_heat * perimeter * (T_heat_C - T_C)
    # Get heat capacity flow (avoid 0)
    Cp_flow = torch.clamp(
        F_CH3OH * Cp_CH3OH + F_CO * Cp_CO + F_H2 * Cp_H2, min=1e-9
    )
    # Energy balance: temperature gradient wrt z
    delta_H_T = reaction_enthalpy(T_C)  # [J/mol] at local T (torch-safe)
    dT = (q_transfer - delta_H_T * rpl) / Cp_flow

    F_total = torch.clamp(F_CH3OH + F_CO + F_H2, min=1e-12)  # total molar flow
    mass_flow = F_CH3OH * MW_CH3OH + F_CO * MW_CO + F_H2 * MW_H2  # total mass flow
    MW_mix = mass_flow / F_total  # average molecular weight
    T_abs = T_C + 273.15  # convert C to K
    rho = P * MW_mix / (R * T_abs)  # gas density
    velocity = mass_flow / torch.clamp(rho * Ac, min=1e-12)  # mixture velocity
    mu = mu_gas  # constant gas viscosity
    viscous = 150.0 * mu * (1.0 - epsilon) ** 2 / (epsilon ** 3 * dp ** 2) * velocity  # viscous term
    inertial = 1.75 * rho * (1.0 - epsilon) / (epsilon ** 3 * dp) * velocity ** 2  # inertial term
    dP = -(viscous + inertial)  # pressure drop
    dQ_kW = q_transfer / 1000.0  # heat in kW

    return {
        "dF_CH3OH": -rpl,
        "dF_CO": rpl,
        "dF_H2": 2.0 * rpl,
        "dT_C": dT,
        "dP": dP,
        "dQ_kW": dQ_kW,
    }


def autodiff_dz(states, z):
    # Compute dz (derivative with respect to z) for all main state variables using autograd
    grads = {}
    for key in ["F_CH3OH", "F_CO", "F_H2", "T_C", "P", "Q_kW"]:
        # Get gradient of this state with respect to z
        g = torch.autograd.grad(
            states[key], z, grad_outputs=torch.ones_like(states[key]),
            create_graph=True, retain_graph=True,
        )[0]
        grads[key] = g
    # Return gradients for all keys
    return grads


print("Physics residual helpers ready.")


## 6. Batch builder + PINN loss

For each operating case we sample:
- collocation `z` points → physics residual
- measurement `z` points → data loss vs Excel
#
**Description of this step:**
# 
In this step, we construct the batch processing logic and define
the physics-informed neural network (PINN) loss function for each operating case.
Specifically, for each set of input conditions (operating case), we:
- Sample collocation points along the reactor length where we will enforce the governing physics equations by minimizing the residuals (physics loss)
- Evaluate the model at measurement locations where real or simulated data is available, adding a data loss term by comparing model predictions to given values (e.g., from Excel)
These components are combined in a loss function that enables the PINN to learn both from physics constraints and available measurement data.



In [ ]:
def make_case_tensors(row):
    # Prepare input tensor (operating conditions)
    u = torch.tensor(
        [[row["T_in_C"], row["P_in_bar_g"], row["F_in_kgmole_h"]]],
        dtype=torch.float32
    )
    # Prepare output tensor (expected output values)
    y_data = torch.tensor(
        [[row[c] for c in OUTPUT_COLS]],
        dtype=torch.float32
    )
    return u, y_data


def pinn_loss_for_case(model, u, y_data, n_colloc=20):
    # Sample points along the reactor for enforcing physics
    z_col = torch.rand(n_colloc, 1) * L
    z_col.requires_grad_(True)
    u_col = u.repeat(n_colloc, 1)

    # Forward pass: get model outputs at sampled points
    states = model(z_col, u_col)
    # Compute physical parameters from states
    eta = model.eta(states["u_n"])
    # Calculate ODE right-hand-side values
    rhs = ode_rhs(states, eta)
    # Calculate model derivatives w.r.t. z
    d_pred = autodiff_dz(states, z_col)

    # Scale residuals
    # Physics loss = sum of squared, normalized residuals for each ODE/component
    L_phys = (
        torch.mean(((d_pred["F_CH3OH"] - rhs["dF_CH3OH"]) / F_REF) ** 2)      # methanol flow
        + torch.mean(((d_pred["F_CO"] - rhs["dF_CO"]) / F_REF) ** 2)          # CO flow
        + torch.mean(((d_pred["F_H2"] - rhs["dF_H2"]) / F_REF) ** 2)          # H2 flow
        + torch.mean(((d_pred["T_C"] - rhs["dT_C"]) / 50.0) ** 2)             # temperature
        + torch.mean(((d_pred["P"] - rhs["dP"]) / 1e4) ** 2)                  # pressure
        + torch.mean(((d_pred["Q_kW"] - rhs["dQ_kW"]) / Q_REF) ** 2)          # heat duty
    )

    # --- data at measurement positions ---
    # Create tensor for measurement positions (where we have real data)
    z_m = torch.tensor(MEASUREMENT_POSITIONS_M, dtype=torch.float32).view(-1, 1)
    z_m.requires_grad_(True)
    # Repeat input conditions for each measurement point
    u_m = u.repeat(len(MEASUREMENT_POSITIONS_M), 1)
    # Predict states at measurement points
    st_m = model(z_m, u_m)

    # Compare predicted and true temperatures at measurement locations
    T_pred = st_m["T_C"].view(-1)
    T_true = y_data[0, 0:5]
    # Compute loss for temperature predictions
    L_T = torch.mean(((T_pred - T_true) / 50.0) ** 2)

    # outlet point z=L
    # Evaluate model at reactor outlet (z = L)
    z_L = torch.tensor([[L]], dtype=torch.float32, requires_grad=True)
    st_L = model(z_L, u)

    # Calculate outlet pressure (bar gauge)
    P_out_bar_g = st_L["P"] / 1.0e5 - P_ATM_BAR
    # Calculate conversion percentage at outlet
    X_pct = (st_L["F_in"] - st_L["F_CH3OH"]) / st_L["F_in"] * 100.0
    # Get outlet heat duty
    Q_kW = st_L["Q_kW"]

    # Compute squared errors for each output at outlet
    L_P = ((P_out_bar_g - y_data[0, 5]) / 1.0) ** 2
    L_Q = ((Q_kW - y_data[0, 6]) / 1.0) ** 2
    L_X = ((X_pct - y_data[0, 7]) / 1.0) ** 2
    
    # Total data loss (temperature + outlet values)
    L_data = L_T + L_P.mean() + L_Q.mean() + L_X.mean()

    # BC already hard-coded; tiny soft BC check at z=0
    # Evaluate model at reactor inlet (z=0)
    z0 = torch.zeros(1, 1, requires_grad=True)
    st0 = model(z0, u)
    # Soft boundary condition loss at reactor inlet
    L_bc = (
        ((st0["T_C"] - u[0, 0]) / 50.0) ** 2    # Enforce T = T_in at z=0
        + ((st0["F_CO"]) / F_REF) ** 2          # CO flow should be small at z=0
        + ((st0["Q_kW"]) / Q_REF) ** 2          # Heat duty small at z=0
    ).mean()

    # Weights: data-dominant (Excel T/Q disagree with wall-heat ODE), light physics
    # Calculate weighted sum of losses
    loss = 1.0 * L_data + 0.05 * L_phys + 0.1 * L_bc
    # Return each loss part for logging/monitoring
    parts = {
        "L_data": float(L_data.detach()),
        "L_phys": float(L_phys.detach()),
        "L_bc": float(L_bc.detach()),
    }
    return loss, parts


print("Loss function ready.")

## 7. Train the PINN

Trains on a subset of operating cases for speed. Increase `N_TRAIN_CASES` / `EPOCHS` for higher accuracy.

In [ ]:
# Practical defaults; raise for better accuracy (e.g. 80 cases / 250 epochs)
N_TRAIN_CASES = 40   # number of training cases
EPOCHS = 120         # number of training epochs
BATCH_CASES = 8      # batch size (cases per batch)
N_COLLOC = 16        # number of collocation points per case (physics sampling)

# Pick random cases from training data
train_cases = train_df.sample(n=min(N_TRAIN_CASES, len(train_df)), random_state=42).reset_index(drop=True)
# Convert each row to tensors for model input
case_data = [make_case_tensors(row) for _, row in train_cases.iterrows()]

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)  # Adam optimizer with learning rate 0.001
history = []  # Store training progress/losses

print(f"Training PINN on {len(case_data)} cases × {EPOCHS} epochs...")
for epoch in range(1, EPOCHS + 1):
    model.train()  # set model to training mode
    idx = np.random.permutation(len(case_data))  # shuffle cases
    total = 0.0  # total loss for epoch
    n_batches = 0  # batch counter
    sum_parts = {"L_data": 0.0, "L_phys": 0.0, "L_bc": 0.0}  # to accumulate loss parts

    for s in range(0, len(idx), BATCH_CASES):
        batch_ids = idx[s:s + BATCH_CASES]  # mini-batch indices
        optimizer.zero_grad()  # reset gradients
        loss_batch = 0.0  # loss sum for this batch
        for j in batch_ids:
            u, y = case_data[j]  # get input and target
            loss_j, parts = pinn_loss_for_case(model, u, y, n_colloc=N_COLLOC)  # get loss and its components
            loss_batch += loss_j  # accumulate batch loss
            for k in sum_parts:
                sum_parts[k] += parts[k]  # add up parts for monitoring
        loss_batch = loss_batch / len(batch_ids)  # average batch loss
        loss_batch.backward()  # backpropagate
        torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)  # gradient clipping
        optimizer.step()  # update weights
        total += float(loss_batch.detach())
        n_batches += 1

    avg = total / max(n_batches, 1)  # average loss for epoch
    scale = 1.0 / max(n_batches * BATCH_CASES, 1)  # average for monitoring
    rec = {"epoch": epoch, "loss": avg,
           "L_data": sum_parts["L_data"] * scale,
           "L_phys": sum_parts["L_phys"] * scale,
           "L_bc": sum_parts["L_bc"] * scale}
    history.append(rec)
    if epoch % 20 == 0 or epoch == 1:
        print(
            f"epoch {epoch:4d} | loss={avg:.4e} | "
            f"data={rec['L_data']:.3e} phys={rec['L_phys']:.3e} bc={rec['L_bc']:.3e}"
        )

# Plot loss history
hist_df = pd.DataFrame(history)
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(hist_df["epoch"], hist_df["loss"], label="total")
ax.plot(hist_df["epoch"], hist_df["L_data"], label="data")
ax.plot(hist_df["epoch"], hist_df["L_phys"], label="physics")
ax.set_xlabel("Epoch")
ax.set_ylabel("Loss")
ax.set_yscale("log")
ax.set_title("PINN training loss")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
print("Training finished.")
print("Tip: increase N_TRAIN_CASES / EPOCHS for better accuracy.")


## 8. Digital twin prediction from the PINN

In [ ]:
@torch.no_grad()
def predict_outputs(T_in_C, P_in_bar_g, F_in_kgmole_h, n_profile=80):
    """PINN digital twin: 3 inputs → 8 Excel-style outputs (+ profile)."""
    model.eval()
    u = torch.tensor([[T_in_C, P_in_bar_g, F_in_kgmole_h]], dtype=torch.float32)

    z_m = torch.tensor(MEASUREMENT_POSITIONS_M, dtype=torch.float32).view(-1, 1)
    st_m = model(z_m, u.repeat(len(MEASUREMENT_POSITIONS_M), 1))
    T_axial = st_m["T_C"].cpu().numpy().reshape(-1)

    z_L = torch.tensor([[L]], dtype=torch.float32)
    st_L = model(z_L, u)
    P_out_bar_g = float(st_L["P"].cpu() / 1e5 - P_ATM_BAR)
    Q_kW = float(st_L["Q_kW"].cpu())
    X_pct = float(((st_L["F_in"] - st_L["F_CH3OH"]) / st_L["F_in"] * 100.0).cpu())
    eta = float(model.eta(st_L["u_n"]).cpu())

    z_prof = torch.linspace(0, L, n_profile).view(-1, 1)
    st_p = model(z_prof, u.repeat(n_profile, 1))

    out = {
        "T_0.2m_C": float(T_axial[0]),
        "T_0.4m_C": float(T_axial[1]),
        "T_0.6m_C": float(T_axial[2]),
        "T_0.8m_C": float(T_axial[3]),
        "T_1m_C": float(T_axial[4]),
        "P_out_bar_g": P_out_bar_g,
        "Q_total_kW": Q_kW,
        "X_overall_pct": max(X_pct, 0.0),
        "eta": eta,
        "z_m": z_prof.cpu().numpy().reshape(-1),
        "T_profile_C": st_p["T_C"].cpu().numpy().reshape(-1),
    }
    return out


demo = train_cases.iloc[0]
pred = predict_outputs(demo["T_in_C"], demo["P_in_bar_g"], demo["F_in_kgmole_h"])
smoke = pd.DataFrame({
    "Output": OUTPUT_COLS,
    "Excel": [demo[c] for c in OUTPUT_COLS],
    "PINN twin": [pred[c] for c in OUTPUT_COLS],
})
smoke["Abs error"] = (smoke["PINN twin"] - smoke["Excel"]).abs()
print(f"Smoke test (eta≈{pred['eta']:.3e}):")
display(smoke.round(4))

## 9. Validate on held-out test cases

In [ ]:
N_TEST_EVAL = min(60, len(test_df))
eval_df = test_df.head(N_TEST_EVAL)

Y_true, Y_pred = [], []
for _, row in eval_df.iterrows():
    p = predict_outputs(row["T_in_C"], row["P_in_bar_g"], row["F_in_kgmole_h"])
    Y_true.append([row[c] for c in OUTPUT_COLS])
    Y_pred.append([p[c] for c in OUTPUT_COLS])

Y_true = np.asarray(Y_true)
Y_pred = np.asarray(Y_pred)

rows = []
for i, col in enumerate(OUTPUT_COLS):
    rows.append({
        "Output": col,
        "MAE PINN": mean_absolute_error(Y_true[:, i], Y_pred[:, i]),
        "R2 PINN": r2_score(Y_true[:, i], Y_pred[:, i]),
    })
metrics = pd.DataFrame(rows)
display(metrics.round(4))

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].scatter(Y_true[:, 7], Y_pred[:, 7], alpha=0.6, s=20)
lim = max(Y_true[:, 7].max(), Y_pred[:, 7].max(), 1e-6) * 1.1
axes[0].plot([0, lim], [0, lim], "k--", lw=1)
axes[0].set_xlabel("Excel conversion (%)")
axes[0].set_ylabel("PINN conversion (%)")
axes[0].set_title("Overall conversion")
axes[0].grid(True, alpha=0.3)

axes[1].scatter(Y_true[:, 4], Y_pred[:, 4], alpha=0.6, s=20)
t0 = min(Y_true[:, 4].min(), Y_pred[:, 4].min()) - 5
t1 = max(Y_true[:, 4].max(), Y_pred[:, 4].max()) + 5
axes[1].plot([t0, t1], [t0, t1], "k--", lw=1)
axes[1].set_xlabel("Excel Temp_1m (°C)")
axes[1].set_ylabel("PINN Temp_1m (°C)")
axes[1].set_title("Outlet temperature")
axes[1].grid(True, alpha=0.3)

fig.suptitle(f"PINN twin vs Excel (n={N_TEST_EVAL} test cases)")
fig.tight_layout()
plt.show()

---

## 10. Interactive digital twin

Enter 3 plant inputs. Type `q` to quit.

In [ ]:
def _ask_float(prompt, default=None):
    hint = f" [{default}]" if default is not None else ""
    while True:
        raw = input(f"{prompt}{hint}: ").strip()
        if raw.lower() in {"q", "quit", "exit"}:
            return None
        if raw == "" and default is not None:
            return float(default)
        try:
            return float(raw)
        except ValueError:
            print("  Enter a number (or q to quit).")


def run_digital_twin_interactive():
    print("\n" + "=" * 62)
    print("  METHANOL REACTOR — PINN DIGITAL TWIN")
    print("  NN(z,u) + ODE residual loss + Excel data loss")
    print("=" * 62)
    print("  Press Enter for defaults. Type q to stop.\n")

    while True:
        T_in_C = _ask_float("Inlet temperature [°C]", default=200.0)
        if T_in_C is None:
            print("Digital twin stopped."); break
        P_in_bar_g = _ask_float("Inlet pressure [bar_g]", default=0.4)
        if P_in_bar_g is None:
            print("Digital twin stopped."); break
        F_in_kgmole_h = _ask_float("Methanol molar flow [kgmole/h]", default=0.1)
        if F_in_kgmole_h is None:
            print("Digital twin stopped."); break
        if F_in_kgmole_h <= 0:
            print("  Flow must be > 0.\n"); continue

        try:
            pred = predict_outputs(T_in_C, P_in_bar_g, F_in_kgmole_h)
        except Exception as exc:
            print(f"  Prediction failed: {exc}\n"); continue

        print("\n" + "-" * 62)
        print("INPUTS")
        print(f"  T_in={T_in_C:.2f} °C | P_in={P_in_bar_g:.4f} bar_g | F={F_in_kgmole_h:.4f} kgmole/h")
        print(f"  Learned eta(u) ≈ {pred['eta']:.4e}")
        print("\nPINN TWIN OUTPUTS")
        for c in OUTPUT_COLS:
            print(f"  {c:16s}: {pred[c]:.4f}")
        print("-" * 62)

        fig, ax = plt.subplots(figsize=(7, 4))
        ax.plot(pred["z_m"], pred["T_profile_C"], lw=2, label="PINN T(z)")
        ax.scatter(
            MEASUREMENT_POSITIONS_M,
            [pred[c] for c in TEMP_COLS],
            color="crimson", zorder=5, label="Sampled T",
        )
        ax.axhline(T_heat_C, color="gray", ls="--", alpha=0.6, label=f"T_heat={T_heat_C:.0f} °C")
        ax.set_xlabel("Reactor length (m)")
        ax.set_ylabel("Temperature (°C)")
        ax.set_title("PINN digital twin — axial temperature")
        ax.legend(); ax.grid(True, alpha=0.3)
        plt.tight_layout(); plt.show()

        again = input("\nRun another case? [Y/n]: ").strip().lower()
        if again in {"n", "no", "q", "quit"}:
            print("Digital twin stopped."); break


run_digital_twin_interactive()